In [ ]:
"""
================================================================================
OGGM PHASE 2 - REGIONAL CLIMATE-CATEGORY ANALYSIS
  Andean glaciers grouped by climate zone (Tropical / Arid / Temperate)

================================================================================
OUTPUTS (saved to ./{Category}_outputs/)
================================================================================

  Per-GCM-per-SSP checkpoints (resumable):
  - run_output_CMIP6_{gcm}_{ssp}.nc   (one per of 36 runs)
  - run_output_spinup_historical.nc
  - run_output_hydro_2000_2019.nc

  Historical validation (2000-2019):
  - {cat}_historical_2000_2019.png     (6-panel regional figure)
  - {cat}_area_volume_2000_2019.csv
  - {cat}_mass_balance_2001_2019.csv
  - {cat}_runoff_2000_2019.csv

  Future projections (2020-2100, 9-GCM x 4 SSP ensemble):
  - fig_volume_change.png     (% of 2020, regional sum, ensemble mean +/- 1 sd)
  - fig_area_change.png       (km^2)
  - fig_mass_balance.png      (B = dV * 0.850 / A_RGI,  Huss 2013)
  - fig_runoff.png            (Mt yr-1, sum of 4 hydro components)
  - fig_thickness.png         (V / A, m)
  - {cat}_projection_annual.csv          (year x SSP, ensemble mean + sd)
  - {cat}_projection_summary.csv         (per SSP: V2020/V2100/peak runoff/...)
  - {cat}_topN_volume.csv                (top-N glaciers, volume trajectories)
  - {cat}_glacier_inventory.csv          (RGI ID, area, top-N rank, etc.)
"""

# ---------------------------------------------------------------------------
# 1. Imports + OGGM Hub setup
# ---------------------------------------------------------------------------
import os
import time
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import oggm.cfg as cfg
from oggm import utils, workflow, tasks, DEFAULT_BASE_URL
from oggm.shop import gcm_climate

# ===========================================================================
# CATEGORY REGISTRY 
# ===========================================================================
CATEGORIES = {
    'Tropical':  {
        'label':    'Tropical Andes',
        'csv_path': 'tropical_rgi_ids.csv',
    },
    'Arid':      {
        'label':    'Arid Andes',
        'csv_path': 'arid_rgi_ids.csv',
    },
    'Temperate': {
        'label':    'Temperate Andes',
        'csv_path': 'temperate_rgi_ids.csv',
    },
}

# === CHANGE THIS LINE TO SWITCH CATEGORIES =================================
CATEGORY = 'Temperate'       # must match a name in CATEGORY REGISTRY 
# ===========================================================================

# Historical validation window (Hugonnet et al. 2021 calibration period)
YS = 2000
YE = 2019

TOP_N = 20

# ---------------------------------------------------------------------------
# 2. Load RGI IDs + initialise OGGM
# ---------------------------------------------------------------------------
cat_info  = CATEGORIES[CATEGORY]
label     = cat_info['label']
csv_path  = cat_info['csv_path']

if not os.path.exists(csv_path):
    raise FileNotFoundError(
        f"RGI ID file not found: {csv_path}\n"
        f"Please create '{csv_path}' next to this notebook, with one\n"
        f"RGI60-... identifier per line (an optional 'rgi_id' header is fine)."
    )

# Read RGI IDs, cope with or without a header
df_ids = pd.read_csv(csv_path, header=None)
if str(df_ids.iloc[0, 0]).lower().startswith('rgi') is False:
    df_ids = pd.read_csv(csv_path)
    id_col = df_ids.columns[0]
    rgi_ids = df_ids[id_col].astype(str).str.strip().tolist()
else:
    rgi_ids = df_ids.iloc[:, 0].astype(str).str.strip().tolist()
# Filter out blanks/duplicates
rgi_ids = [r for r in rgi_ids if r.startswith('RGI60-')]
rgi_ids = list(dict.fromkeys(rgi_ids))
print(f'Loaded {len(rgi_ids)} RGI IDs from {csv_path}')

cfg.initialize(logging_level='WARNING')
cfg.PATHS['working_dir'] = utils.gettempdir(
    f'OGGM_{CATEGORY}_regional', reset=False)  # keep across re-runs
cfg.PARAMS['min_ice_thick_for_length'] = 1
cfg.PARAMS['store_model_geometry']     = True
cfg.PARAMS['continue_on_error']        = True
cfg.PARAMS['use_multiprocessing']      = True   # critical for 500 glaciers

OUTDIR = os.path.join(os.getcwd(), f'{CATEGORY}_outputs')
os.makedirs(OUTDIR, exist_ok=True)
print(f'Output folder: {OUTDIR}')
print(f'Working directory: {cfg.PATHS["working_dir"]}')

# Load preprocessed directories 
gdirs = workflow.init_glacier_directories(
    rgi_ids,
    from_prepro_level=5,
    prepro_base_url=DEFAULT_BASE_URL,
)
# Sort by area descending 
gdirs = sorted(gdirs, key=lambda g: g.rgi_area_km2, reverse=True)
print(f'Initialised {len(gdirs)} glacier directories for {label}')
print(f'  Total area: {sum(g.rgi_area_km2 for g in gdirs):.2f} km^2')
print(f'  Largest: {gdirs[0].rgi_id} ({gdirs[0].rgi_area_km2:.2f} km^2)')
print(f'  Smallest: {gdirs[-1].rgi_id} ({gdirs[-1].rgi_area_km2:.3f} km^2)')

# Record top-N largest glaciers
top_n_ids = [g.rgi_id for g in gdirs[:TOP_N]]
print(f'\nTop {TOP_N} largest glaciers tracked individually:')
for i, g in enumerate(gdirs[:TOP_N]):
    print(f'  {i+1:2d}. {g.rgi_id}  {g.rgi_area_km2:7.2f} km^2')

# Save glacier inventory
pd.DataFrame({
    'rgi_id':       [g.rgi_id for g in gdirs],
    'area_km2':     [g.rgi_area_km2 for g in gdirs],
    'rgi_date':     [g.rgi_date for g in gdirs],
    'cenlat':       [g.cenlat for g in gdirs],
    'cenlon':       [g.cenlon for g in gdirs],
    'top_n_rank':   [i + 1 if i < TOP_N else np.nan
                     for i in range(len(gdirs))],
}).to_csv(os.path.join(OUTDIR, f'{CATEGORY}_glacier_inventory.csv'),
         index=False)

# ---------------------------------------------------------------------------
# 3. CMIP6 projection runs (9 GCMs x 4 SSPs) - resumable
# ---------------------------------------------------------------------------
GCMS = [
    'BCC-CSM2-MR', 'CAMS-CSM1-0', 'CESM2', 'EC-Earth3-Veg',
    'FGOALS-f3-L', 'GFDL-ESM4', 'MPI-ESM1-2-HR',
    'MRI-ESM2-0', 'NorESM2-MM',
]
SSPS = ['ssp126', 'ssp245', 'ssp370', 'ssp585']

bp_tmpl = ('https://cluster.klima.uni-bremen.de/~oggm/cmip6/GCM/'
           '{gcm}/{gcm}_{ssp}_r1i1p1f1_pr.nc')
bt_tmpl = ('https://cluster.klima.uni-bremen.de/~oggm/cmip6/GCM/'
           '{gcm}/{gcm}_{ssp}_r1i1p1f1_tas.nc')

def checkpoint_path(fid):
    """Compiled NetCDF path for a given run filesuffix."""
    return os.path.join(OUTDIR, f'run_output{fid}.nc')

print(f'\n{"="*72}')
print(f'Starting CMIP6 runs: {len(GCMS)} GCMs x {len(SSPS)} SSPs '
      f'= {len(GCMS)*len(SSPS)} runs')
print(f'Completed runs found on disk will be skipped.')
print(f'{"="*72}')

t0 = time.time()
for gcm in GCMS:
    for ssp in SSPS:
        fid = f'_CMIP6_{gcm}_{ssp}'
        out_nc = checkpoint_path(fid)
        if os.path.exists(out_nc):
            print(f'>>> SKIP   {gcm} {ssp}  (already on disk)')
            continue
        print(f'>>> Processing {gcm} {ssp} ...')
        try:
            ft = utils.file_downloader(bt_tmpl.format(gcm=gcm, ssp=ssp))
            fp = utils.file_downloader(bp_tmpl.format(gcm=gcm, ssp=ssp))
            if ft is None or fp is None:
                print('    skipped (CMIP6 file not on Bremen server)')
                continue
            workflow.execute_entity_task(
                gcm_climate.process_cmip_data, gdirs,
                filesuffix=fid,
                fpath_temp=ft,
                fpath_precip=fp,
            )
            workflow.execute_entity_task(
                tasks.run_with_hydro, gdirs,
                run_task=tasks.run_from_climate_data,
                climate_filename='gcm_data',
                climate_input_filesuffix=fid,
                init_model_filesuffix='_spinup_historical',
                output_filesuffix=fid,
                store_monthly_hydro=False,
            )
            
            utils.compile_run_output(gdirs, path=out_nc,
                                     input_filesuffix=fid)
            elapsed = (time.time() - t0) / 60.0
            print(f'    done. Cumulative wall time: {elapsed:.1f} min')
        except Exception as e:
            print(f'    FAILED: {e}')

print(f'\nAll projection runs complete. Total wall time: '
      f'{(time.time()-t0)/60.0:.1f} min')

# ---------------------------------------------------------------------------
# 4. Historical runs - resumable
# ---------------------------------------------------------------------------
HIST_SUFFIX  = '_spinup_historical'
HYDRO_SUFFIX = f'_hydro_{YS}_{YE}'

hist_nc  = checkpoint_path(HIST_SUFFIX)
hydro_nc = checkpoint_path(HYDRO_SUFFIX)

if not os.path.exists(hist_nc):
    print('\n>>> Compiling historical spinup run...')
    utils.compile_run_output(gdirs, path=hist_nc,
                             input_filesuffix=HIST_SUFFIX)
else:
    print('\n>>> Historical spinup already compiled - skipping')

HYDRO_OK = False
if not os.path.exists(hydro_nc):
    print(f'>>> Running historical hydro {YS}-{YE} ...')
    try:
        workflow.execute_entity_task(
            tasks.run_with_hydro, gdirs,
            run_task=tasks.run_from_climate_data,
            ys=YS, ye=YE,
            init_model_filesuffix=HIST_SUFFIX,
            init_model_yr=YS,
            ref_area_from_y0=True,
            output_filesuffix=HYDRO_SUFFIX,
            store_monthly_hydro=False,   # big + slow for 500 glaciers
        )
        utils.compile_run_output(gdirs, path=hydro_nc,
                                 input_filesuffix=HYDRO_SUFFIX)
        HYDRO_OK = True
        print('    done.')
    except Exception as e:
        print(f'    FAILED (runoff panels will be blank): {e}')
else:
    print('>>> Historical hydro already compiled - skipping')
    HYDRO_OK = True

# ---------------------------------------------------------------------------
# 5. Constants + plotting 
# ---------------------------------------------------------------------------
ICE_DENSITY = 0.850     

SSP_COLOURS = {
    'ssp126': '#1f77b4', 'ssp245': '#2ca02c',
    'ssp370': '#ff7f0e', 'ssp585': '#d62728',
}
SSP_LABELS = {
    'ssp126': 'SSP1-2.6', 'ssp245': 'SSP2-4.5',
    'ssp370': 'SSP3-7.0', 'ssp585': 'SSP5-8.5',
}

ANNUAL_C = '#1f4e79'
MEAN5_C  = 'green'
MEAN11_C = 'orange'
OBS_C    = 'red'

def roll(vals, w):
    return pd.Series(vals).rolling(window=w, center=True,
                                   min_periods=1).mean().values

def styled_plot(ax, df, ssp):
    c = SSP_COLOURS[ssp]
    ax.fill_between(df['time'], df['mean'] - df['std'],
                    df['mean'] + df['std'], color=c, alpha=0.15)
    ax.plot(df['time'], df['mean'], color=c, lw=2, label=SSP_LABELS[ssp])

def get_years(time_coord):
    try:
        return time_coord.dt.year.values.tolist()
    except AttributeError:
        return [int(str(t)[:4]) for t in time_coord.values]

# ---------------------------------------------------------------------------
# 6. Load projection ensemble into memory
# ---------------------------------------------------------------------------
print('\nAssembling projection ensemble...')

def load_run(gcm, ssp):
    """Open one GCMxSSP NetCDF, trim to 2020-2100, select regional sums."""
    fid = f'_CMIP6_{gcm}_{ssp}'
    p   = checkpoint_path(fid)
    if not os.path.exists(p):
        return None
    ds = xr.open_dataset(p).sel(time=slice(2020, 2100))
    return ds

var_list = ['volume', 'area',
            'melt_on_glacier',  'melt_off_glacier',
            'liq_prcp_on_glacier', 'liq_prcp_off_glacier']

regional = {v: {} for v in var_list}        
top_n_volume = {ssp: {} for ssp in SSPS}    

for ssp in SSPS:
    for v in var_list:
        regional[v][ssp] = {}
    for gcm in GCMS:
        ds = load_run(gcm, ssp)
        if ds is None:
            print(f'  skip {gcm} {ssp} (checkpoint missing)')
            continue
        for v in var_list:
            if v in ds.data_vars:
                regional[v][ssp][gcm] = ds[v].sum('rgi_id').load()
        # Top-N per-glacier volume
        if 'volume' in ds.data_vars:
            sel_ids = [rid for rid in top_n_ids if rid in ds.rgi_id.values]
            if sel_ids:
                top_n_volume[ssp][gcm] = ds['volume'].sel(
                    rgi_id=sel_ids).load()
        ds.close()

def stack(var, ssp):
    """Stack dict of {gcm: DataArray(time)} into DataArray(time, gcm)."""
    d = regional[var][ssp]
    if not d:
        return None
    gcm_names = list(d.keys())
    arr = np.stack([d[g].values for g in gcm_names], axis=1)
    return xr.DataArray(
        arr,
        dims=('time', 'gcm'),
        coords={'time': list(d.values())[0].time.values,
                'gcm':  gcm_names},
        name=var,
    )

def ensemble_df(var, ssp, factor=1.0):
    da = stack(var, ssp)
    if da is None:
        return None
    return pd.DataFrame({
        'time': da.time.values,
        'mean': (da.mean('gcm').values * factor),
        'std':  (da.std('gcm').values  * factor),
    })

# ===========================================================================
# 7. HISTORICAL ANALYSIS (2000-2019)
# ===========================================================================
print('\n' + '=' * 72)
print(f'HISTORICAL ANALYSIS - {label} {YS}-{YE}')
print('=' * 72)

DS_HIST = xr.open_dataset(hist_nc)
all_years = get_years(DS_HIST.time)
keep      = [i for i, y in enumerate(all_years) if YS <= y <= YE]
ds_hist   = DS_HIST.isel(time=keep)

volume = ds_hist.volume.sum(dim='rgi_id')
area   = ds_hist.area.sum(dim='rgi_id')

years_av  = get_years(volume.time)
vol_vals  = volume.values
area_vals = area.values
area_km2  = area_vals / 1e6

area_loss_cum = area_km2[0] - area_km2
area_pct_loss = (area_loss_cum / area_km2[0]) * 100
vol_pct       = (vol_vals / vol_vals[0]) * 100
vol_loss_pct  = 100 - vol_pct[-1]
area_loss_total = area_km2[0] - area_km2[-1]

coeffs     = np.polyfit(years_av, area_km2, 1)
area_rate  = coeffs[0]
area_trend = np.polyval(coeffs, years_av)

area_mean    = 0.5 * (area_vals[:-1] + area_vals[1:])
mass_balance = (np.diff(vol_vals) * ICE_DENSITY) / area_mean
years_mb     = years_av[1:]

# Historical runoff from hydro run
gr_vals = tr_vals = years_runoff = None
if HYDRO_OK and os.path.exists(hydro_nc):
    ds_h = xr.open_dataset(hydro_nc)
    
    ds_h = ds_h.isel(time=slice(0, -1))
    GLACIER_RUNOFF_VARS = ['melt_on_glacier', 'liq_prcp_on_glacier']
    ALL_RUNOFF_VARS     = GLACIER_RUNOFF_VARS + [
                          'melt_off_glacier', 'liq_prcp_off_glacier']
    try:
        gr = (ds_h[GLACIER_RUNOFF_VARS].to_array('c')
              .sum('c').clip(min=0).sum('rgi_id') * 1e-9)   # kg -> Mt
        tr = (ds_h[ALL_RUNOFF_VARS].to_array('c')
              .sum('c').clip(min=0).sum('rgi_id') * 1e-9)
        gr_vals = gr.values
        tr_vals = tr.values
        years_runoff = get_years(gr.time)
    except Exception as e:
        print(f'  Runoff extraction failed: {e}')
    ds_h.close()

area_5,  area_11  = roll(area_km2,      5), roll(area_km2,      11)
loss_5,  loss_11  = roll(area_loss_cum, 5), roll(area_loss_cum, 11)
vpct_5,  vpct_11  = roll(vol_pct,       5), roll(vol_pct,       11)
mb_5,    mb_11    = roll(mass_balance,  5), roll(mass_balance,  11)
if gr_vals is not None:
    gr_5, gr_11 = roll(gr_vals, 5), roll(gr_vals, 11)
    tr_5, tr_11 = roll(tr_vals, 5), roll(tr_vals, 11)

# 6-panel historical figure
fig, axes = plt.subplots(3, 2, figsize=(16, 14))
axes = axes.flatten()

def _fmt(ax, x, tick_step=2):
    ax.set_xlim(min(x) - 0.5, max(x) + 0.5)
    ax.xaxis.set_major_locator(mticker.MultipleLocator(tick_step))
    ax.xaxis.set_minor_locator(mticker.MultipleLocator(1))
    ax.tick_params(axis='x', rotation=45)
    ax.set_xlabel('Year')

def _panel(ax, x, y, y5, y11, title, ylabel, hline=None,
           tick_step=2):
    ax.plot(x, y,   color=ANNUAL_C, lw=1.2, label='Annual')
    ax.plot(x, y5,  color=MEAN5_C,  lw=1.5, label='5-yr mean')
    ax.plot(x, y11, color=MEAN11_C, lw=2.5, label='11-yr mean')
    if hline is not None:
        ax.axhline(hline, color='grey', ls='--', lw=1)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=9)
    _fmt(ax, x, tick_step)

# Panel 1: Regional area w/ OLS trend
ax = axes[0]
ax.plot(years_av, area_km2,   color=ANNUAL_C, lw=1.2, label='Annual')
ax.plot(years_av, area_5,     color=MEAN5_C,  lw=1.5, label='5-yr mean')
ax.plot(years_av, area_11,    color=MEAN11_C, lw=2.5, label='11-yr mean')
ax.plot(years_av, area_trend, color=OBS_C,    lw=1.5, ls='--',
        label=f'Linear trend: {area_rate:.3f} km$^2$ a$^{{-1}}$')
ax.set_title(f'{label} - regional area {YS}-{YE} (N = {len(gdirs)})',
             fontsize=11, fontweight='bold')
ax.set_ylabel('Area (km$^2$)')
ax.legend(fontsize=9)
_fmt(ax, years_av)

_panel(axes[1], years_av, area_loss_cum, loss_5, loss_11,
       f'{label} - cumulative area loss {YS}-{YE}',
       f'Area lost from {YS} (km$^2$)', hline=0)

_panel(axes[2], years_av, vol_pct, vpct_5, vpct_11,
       f'{label} - relative volume change {YS}-{YE}',
       f'Volume (% of {YS})', hline=100)

_panel(axes[3], years_mb, mass_balance, mb_5, mb_11,
       f'{label} - area-averaged mass balance {YS+1}-{YE}',
       'Mass balance (m w.e. a$^{-1}$)', hline=0)

if gr_vals is not None:
    _panel(axes[4], years_runoff, gr_vals, gr_5, gr_11,
           f'{label} - glacier runoff {YS}-{YE}', 'Runoff (Mt)')
    _panel(axes[5], years_runoff, tr_vals, tr_5, tr_11,
           f'{label} - total catchment runoff {YS}-{YE}', 'Runoff (Mt)')
else:
    for ax in (axes[4], axes[5]):
        ax.text(0.5, 0.5, 'Historical hydro run failed\nrunoff unavailable',
                ha='center', va='center', transform=ax.transAxes,
                fontsize=11, color='grey')
        ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
hist_fig_path = os.path.join(
    OUTDIR, f'{CATEGORY}_historical_{YS}_{YE}.png')
fig.savefig(hist_fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Historical figure saved: {hist_fig_path}')

# Historical CSV exports
pd.DataFrame({
    'year':                       years_av,
    'area_km2':                   area_km2,
    'area_5yr_km2':               area_5,
    'area_11yr_km2':              area_11,
    'area_trend_km2':             area_trend,
    f'area_loss_from_{YS}_km2':   area_loss_cum,
    f'area_pct_lost_from_{YS}':   area_pct_loss,
    'volume_m3':                  vol_vals,
    f'volume_pct_of_{YS}':        vol_pct,
}).to_csv(os.path.join(OUTDIR, f'{CATEGORY}_area_volume_{YS}_{YE}.csv'),
         index=False)

pd.DataFrame({
    'year':              years_mb,
    'mass_balance_mwea': mass_balance,
    'mass_balance_5yr':  mb_5,
    'mass_balance_11yr': mb_11,
}).to_csv(os.path.join(OUTDIR,
          f'{CATEGORY}_mass_balance_{YS+1}_{YE}.csv'), index=False)

if gr_vals is not None:
    pd.DataFrame({
        'year':                years_runoff,
        'glacier_runoff_Mt':   gr_vals,
        'total_runoff_Mt':     tr_vals,
    }).to_csv(os.path.join(OUTDIR,
             f'{CATEGORY}_runoff_{YS}_{YE}.csv'), index=False)

DS_HIST.close()

# ===========================================================================
# 8. FUTURE PROJECTION FIGURES (2020-2100)
# ===========================================================================
print('\n' + '=' * 72)
print(f'FUTURE PROJECTIONS - {label} 2020-2100')
print('=' * 72)

# Total RGI area (m^2) for mass-balance calc (Aguayo/Cort/Alaska convention)
A_RGI_TOT_M2 = sum(g.rgi_area_km2 for g in gdirs) * 1e6

# --- 8a: Volume change (%, ref = 2020) -------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))
for ssp in SSPS:
    df = ensemble_df('volume', ssp)
    if df is None:
        continue
    ref = df['mean'].iloc[0]
    df['mean'] = df['mean'] / ref * 100.
    df['std']  = df['std']  / ref * 100.
    styled_plot(ax, df, ssp)
    end_m, end_s = df['mean'].iloc[-1], df['std'].iloc[-1]
    ax.text(2101, end_m, f'{end_m:.0f} $\\pm$ {end_s:.0f}%',
            color=SSP_COLOURS[ssp], va='center',
            fontsize=9, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Volume (% of 2020)')
ax.set_title(f'{label} (N = {len(gdirs)}) - '
             f'projected volume change 2020-2100')
ax.legend(fontsize=10, loc='lower left')
ax.grid(alpha=0.3)
ax.set_xlim(2020, 2105)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, 'fig_volume_change.png'), dpi=200)

# --- 8b: Area change (km^2) ------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))
for ssp in SSPS:
    df = ensemble_df('area', ssp, factor=1e-6)  # m^2 -> km^2
    if df is None:
        continue
    styled_plot(ax, df, ssp)
ax.set_xlabel('Year')
ax.set_ylabel('Glacier area (km$^2$)')
ax.set_title(f'{label} - projected area change')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, 'fig_area_change.png'), dpi=200)

# --- 8c: Area-averaged annual mass balance (Huss 2013) ---------------------
fig, ax = plt.subplots(figsize=(10, 6))
for ssp in SSPS:
    vol_da = stack('volume', ssp)
    if vol_da is None:
        continue
    dv = vol_da.diff('time')
    B  = dv * ICE_DENSITY / A_RGI_TOT_M2
    df = pd.DataFrame({
        'time': B.time.values,
        'mean': B.mean('gcm').values,
        'std':  B.std('gcm').values,
    })
    styled_plot(ax, df, ssp)
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('Year')
ax.set_ylabel('Area-averaged mass balance B (m w.e. yr$^{-1}$)')
ax.set_title(f'{label} - area-averaged annual mass balance  '
             r'($B = \Delta V \times 0.850 / A_{RGI}$, Huss 2013)')
ax.legend(fontsize=10, loc='lower left')
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, 'fig_mass_balance.png'), dpi=200)

# --- 8d: Annual runoff (Mt yr-1) ------------------------------------------
RUNOFF_VARS = ['melt_on_glacier', 'melt_off_glacier',
               'liq_prcp_on_glacier', 'liq_prcp_off_glacier']

fig, ax = plt.subplots(figsize=(10, 6))
for ssp in SSPS:
    parts = []
    for v in RUNOFF_VARS:
        st = stack(v, ssp)
        if st is not None:
            parts.append(st)
    if not parts:
        continue
    runoff_da = sum(parts) * 1e-9   # kg -> Mt
    df = pd.DataFrame({
        'time': runoff_da.time.values,
        'mean': runoff_da.mean('gcm').values,
        'std':  runoff_da.std('gcm').values,
    })
    styled_plot(ax, df, ssp)
ax.set_xlabel('Year')
ax.set_ylabel('Total annual runoff (Mt yr$^{-1}$)')
ax.set_title(f'{label} - projected annual runoff\n'
             '(on- + off-glacier melt + liquid precipitation)')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, 'fig_runoff.png'), dpi=200)

# --- 8e: Mean ice thickness (V/A) ------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))
for ssp in SSPS:
    v = stack('volume', ssp)
    a = stack('area',   ssp)
    if v is None or a is None:
        continue
    h = v / a.where(a > 0)
    df = pd.DataFrame({
        'time': h.time.values,
        'mean': h.mean('gcm').values,
        'std':  h.std('gcm').values,
    })
    styled_plot(ax, df, ssp)
ax.set_xlabel('Year')
ax.set_ylabel('Mean ice thickness V/A (m)')
ax.set_title(f'{label} - projected mean ice thickness')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, 'fig_thickness.png'), dpi=200)

plt.show()

# ===========================================================================
# 9. PROJECTION SUMMARY BLOCK
# ===========================================================================
print(f"\n=== {label.upper()} 2020-2100 SUMMARY ===")
print(f"  Glaciers: {len(gdirs)}")
print(f"  Total RGI area: {A_RGI_TOT_M2/1e6:.1f} km^2")
print(f"  GCM ensemble ({len(GCMS)} GCMs, CMIP6):")
for i, g in enumerate(GCMS):
    print(f'     {i+1}. {g}')
print()
print(f"  {'SSP':<10}{'V2020 (km^3)':>14}{'V2100 (km^3)':>15}"
      f"{'% remaining':>16}{'A2100 (km^2)':>15}"
      f"{'Peak runoff (Mt)':>19}{'Peak year':>12}")
print(f"  {'-'*10}{'-'*14}{'-'*15}{'-'*16}{'-'*15}{'-'*19}{'-'*12}")

proj_rows = []
for ssp in SSPS:
    v_da = stack('volume', ssp)
    a_da = stack('area',   ssp)
    if v_da is None or a_da is None:
        continue
    # Runoff sum
    parts = [stack(v, ssp) for v in RUNOFF_VARS]
    parts = [p for p in parts if p is not None]
    r_da  = sum(parts) * 1e-9 if parts else None

    v_mean = v_da.mean('gcm')
    v_std  = v_da.std('gcm')
    a_mean = a_da.mean('gcm')

    v2020     = float(v_mean.sel(time=2020)) * 1e-9
    v2100     = float(v_mean.sel(time=2100)) * 1e-9
    v2100_std = float(v_std .sel(time=2100)) * 1e-9
    a2100     = float(a_mean.sel(time=2100)) * 1e-6
    pct       = v2100 / v2020 * 100 if v2020 > 0 else 0.0
    pct_std   = v2100_std / v2020 * 100 if v2020 > 0 else 0.0

    if r_da is not None:
        r_mean   = r_da.mean('gcm')
        peak_yr  = int(r_mean.idxmax('time').item())
        peak_val = float(r_mean.max('time'))
    else:
        peak_yr, peak_val = -999, float('nan')

    print(f"  {SSP_LABELS[ssp]:<10}"
          f"{v2020:>14.3f}{v2100:>15.3f}"
          f"{pct:>10.1f} $\\pm${pct_std:>4.0f}%"
          f"{a2100:>15.2f}{peak_val:>17.1f}{peak_yr:>14d}")

    dv = v_da.diff('time')
    B  = dv * ICE_DENSITY / A_RGI_TOT_M2
    B_mean = float(B.mean('gcm').mean('time'))
    v_2071_2100 = v_mean.sel(time=slice(2071, 2100)) * 1e-9
    a_2071_2100 = a_mean.sel(time=slice(2071, 2100)) * 1e-6

    proj_rows.append({
        'ssp':                        ssp,
        'V_2020_km3':                 v2020,
        'V_2100_km3':                 v2100,
        'V_2100_km3_std':             v2100_std,
        'pct_remaining_2100':         pct,
        'pct_remaining_2100_std':     pct_std,
        'A_2100_km2':                 a2100,
        'peak_runoff_Mt':             peak_val,
        'peak_runoff_year':           peak_yr,
        'century_mean_mb_mwe_per_yr': B_mean,
        'V_2071_2100_mean_km3':       float(v_2071_2100.mean()),
        'A_2071_2100_mean_km2':       float(a_2071_2100.mean()),
    })

pd.DataFrame(proj_rows).to_csv(
    os.path.join(OUTDIR, f'{CATEGORY}_projection_summary.csv'), index=False)

# Annual ensemble-mean CSV (one row per year x SSP)
rows = []
for ssp in SSPS:
    v = stack('volume', ssp); a = stack('area', ssp)
    if v is None or a is None:
        continue
    parts = [stack(rv, ssp) for rv in RUNOFF_VARS]
    parts = [p for p in parts if p is not None]
    r = sum(parts) if parts else None
    for t in v.time.values:
        row = {
            'ssp':              ssp,
            'year':             int(t),
            'volume_m3_mean':   float(v.sel(time=t).mean('gcm')),
            'volume_m3_std':    float(v.sel(time=t).std('gcm')),
            'area_m2_mean':     float(a.sel(time=t).mean('gcm')),
            'area_m2_std':      float(a.sel(time=t).std('gcm')),
        }
        if r is not None:
            row['runoff_kg_mean'] = float(r.sel(time=t).mean('gcm'))
            row['runoff_kg_std']  = float(r.sel(time=t).std('gcm'))
        rows.append(row)
pd.DataFrame(rows).to_csv(
    os.path.join(OUTDIR, f'{CATEGORY}_projection_annual.csv'), index=False)

# Top-N per-glacier volume trajectories (Alaska MSc Fig. 11 style)
topn_rows = []
for ssp in SSPS:
    for gcm, da in top_n_volume[ssp].items():
        # da: (time, rgi_id)
        for rid in da.rgi_id.values:
            ts = da.sel(rgi_id=rid)
            for t in ts.time.values:
                topn_rows.append({
                    'ssp':        ssp,
                    'gcm':        gcm,
                    'rgi_id':     str(rid),
                    'year':       int(t),
                    'volume_m3':  float(ts.sel(time=t)),
                })
if topn_rows:
    pd.DataFrame(topn_rows).to_csv(
        os.path.join(OUTDIR, f'{CATEGORY}_topN_volume.csv'), index=False)

# ===========================================================================
# 10. EXPORT SUMMARY
# ===========================================================================
EXPORTS = [
    # Historical
    ('Historical 6-panel figure',
     f'{CATEGORY}_historical_{YS}_{YE}.png'),
    ('Historical area+volume CSV',
     f'{CATEGORY}_area_volume_{YS}_{YE}.csv'),
    ('Historical mass balance CSV',
     f'{CATEGORY}_mass_balance_{YS+1}_{YE}.csv'),
    ('Historical runoff CSV',
     f'{CATEGORY}_runoff_{YS}_{YE}.csv'),
    # Projection plots
    ('Projection volume change plot', 'fig_volume_change.png'),
    ('Projection area change plot',   'fig_area_change.png'),
    ('Projection mass balance plot',  'fig_mass_balance.png'),
    ('Projection runoff plot',        'fig_runoff.png'),
    ('Projection thickness plot',     'fig_thickness.png'),
    # Projection tables
    ('Projection annual CSV',         f'{CATEGORY}_projection_annual.csv'),
    ('Projection summary CSV',        f'{CATEGORY}_projection_summary.csv'),
    ('Top-N volume trajectories',     f'{CATEGORY}_topN_volume.csv'),
    ('Glacier inventory',             f'{CATEGORY}_glacier_inventory.csv'),
]

print('\n' + '=' * 72)
print(f'EXPORT SUMMARY - {len(EXPORTS)} files in:')
print(f'   {OUTDIR}')
print('=' * 72)
for lbl, fname in EXPORTS:
    path = os.path.join(OUTDIR, fname)
    status = 'OK ' if os.path.exists(path) else 'MISSING'
    size = (f'{os.path.getsize(path)/1024:.1f} KB'
            if os.path.exists(path) else '')
    print(f'  [{status}] {lbl:32s}  {fname:42s}  {size}')

# Also list the 36+ per-run checkpoint NetCDFs
ckpts = sorted(f for f in os.listdir(OUTDIR) if f.startswith('run_output'))
print(f'\nPer-run checkpoint NetCDFs (resumable cache): {len(ckpts)} files')
for f in ckpts:
    sz = os.path.getsize(os.path.join(OUTDIR, f)) / 1024 / 1024
    print(f'  {f}  ({sz:.1f} MB)')
print('=' * 72)

print('\nTo download from the OGGM Hub:')
print(f'  Open the "{CATEGORY}_outputs" folder in JupyterLab, right-click, '
      '"Download".')
print('\nTo run next category: change CATEGORY variable (top of notebook) '
      'and re-run.')